# Classifying Penguins with Keras

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn import preprocessing
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

In [3]:
! pip install palmerpenguins
from palmerpenguins import load_penguins
penguins = load_penguins()
penguins.head()



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [4]:
penguins = penguins.dropna()
penguins.shape

(333, 8)

In [5]:
# shuffle the data
penguins = penguins.sample(frac=1, random_state=42).reset_index(drop=True)


In [6]:
penguins_x = pd.concat([penguins[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']], pd.get_dummies(penguins['sex'])], axis = 1)
# penguins_x = penguins_x[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'female', 'male']]
penguins_x

,body_mass_g,bill_length_mm,bill_depth_mm,flipper_length_mm,female,male
0,3250.0,39.5,16.7,178.0,True,False
1,3675.0,50.9,17.9,196.0,True,False
2,4000.0,42.1,19.1,195.0,False,True
3,4850.0,46.6,14.2,210.0,True,False
4,4050.0,41.1,18.2,192.0,False,True
...,...,...,...,...,...,...
328,4750.0,49.6,15.0,216.0,False,True
329,3900.0,37.2,19.4,184.0,False,True
330,3200.0,39.7,17.7,193.0,True,False
331,3950.0,45.2,17.8,198.0,True,False


In [7]:
x = penguins_x.values
min_max_scaler = preprocessing.MinMaxScaler()
scaled_penguins_x = pd.DataFrame(min_max_scaler.fit_transform(x), columns=penguins_x.columns)
scaled_penguins_x

,body_mass_g,bill_length_mm,bill_depth_mm,flipper_length_mm,female,male
0,0.152778,0.269091,0.428571,0.101695,1.0,0.0
1,0.270833,0.683636,0.571429,0.406780,1.0,0.0
2,0.361111,0.363636,0.714286,0.389831,0.0,1.0
3,0.597222,0.527273,0.130952,0.644068,1.0,0.0
4,0.375000,0.327273,0.607143,0.338983,0.0,1.0
...,...,...,...,...,...,...
328,0.569444,0.636364,0.226190,0.745763,0.0,1.0
329,0.333333,0.185455,0.750000,0.203390,0.0,1.0
330,0.138889,0.276364,0.547619,0.355932,1.0,0.0
331,0.347222,0.476364,0.559524,0.440678,1.0,0.0


In [8]:
penguins_y = penguins['species']
print(penguins_y)
penguins_y = penguins_y.astype('category').cat.codes.to_numpy()
penguins_y

0         Adelie
1      Chinstrap
2         Adelie
3         Gentoo
4         Adelie
         ...    
328       Gentoo
329       Adelie
330       Adelie
331    Chinstrap
332       Adelie
Name: species, Length: 333, dtype: str


array([0, 1, 0, 2, 0, 1, 1, 2, 2, 2, 0, 0, 1, 0, 1, 0, 0, 2, 0, 1, 0, 0,
       1, 2, 0, 0, 2, 1, 2, 1, 2, 1, 0, 0, 1, 1, 2, 2, 0, 0, 0, 0, 2, 2,
       0, 0, 1, 0, 0, 1, 0, 2, 2, 0, 0, 2, 0, 0, 2, 2, 1, 1, 1, 0, 0, 1,
       0, 2, 0, 1, 0, 0, 2, 1, 2, 2, 0, 0, 0, 2, 0, 0, 2, 0, 1, 2, 0, 1,
       2, 2, 2, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 1, 1, 0, 2, 0, 2, 2, 0, 2,
       0, 1, 0, 2, 2, 2, 0, 2, 0, 2, 0, 2, 1, 0, 0, 1, 0, 0, 0, 2, 0, 0,
       2, 0, 0, 0, 2, 0, 1, 0, 0, 2, 0, 1, 2, 1, 2, 1, 2, 2, 2, 2, 0, 0,
       2, 2, 2, 0, 2, 2, 0, 1, 1, 1, 2, 2, 2, 2, 2, 0, 0, 2, 1, 0, 1, 1,
       0, 0, 0, 0, 1, 0, 2, 1, 0, 2, 2, 0, 1, 0, 1, 0, 2, 0, 2, 0, 0, 0,
       2, 0, 2, 0, 1, 0, 0, 2, 2, 2, 0, 0, 0, 2, 2, 0, 0, 1, 0, 2, 0, 1,
       1, 1, 0, 2, 1, 2, 2, 0, 2, 0, 0, 2, 0, 2, 0, 2, 1, 0, 1, 2, 1, 0,
       2, 2, 2, 0, 0, 0, 2, 2, 2, 1, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 1,
       1, 2, 1, 2, 2, 2, 1, 2, 1, 1, 1, 2, 2, 0, 2, 2, 2, 0, 0, 0, 0, 0,
       2, 0, 2, 0, 0, 2, 2, 0, 0, 1, 2, 1, 0, 1, 2,

In [9]:
#construct the model
inputs = keras.Input(shape=(6,))
x = layers.Dense(7, activation = 'relu')(inputs)
x = layers.Dense(5, activation = 'relu')(x)
x = layers.Dense(3, activation = 'relu')(x)
outputs = layers.Dense(3, activation='sigmoid')(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="penguin_model")

In [10]:
model.summary()

Model: "penguin_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 7)              │            49 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            18 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            12 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 119 (476.00 B)

 Trainable params: 119 (476.00 B)

 Non-trainable params: 0 (0.00 B)

In [12]:
keras.utils.plot_model(model, show_shapes = True)

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [13]:
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

history = model.fit(scaled_penguins_x, penguins_y, batch_size = 64, epochs=100, validation_split=0.1)

scores = model.evaluate(scaled_penguins_x, penguins_y, verbose=2)

Epoch 1/100


c:\Users\ivpri\gsb545\.venv_clean\Lib\site-packages\keras\src\backend\tensorflow\nn.py:1216: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


1/5 ━━━━━━━━━━━━━━━━━━━━ 12s 3s/step - accuracy: 0.1250 - loss: 1.1456

c:\Users\ivpri\gsb545\.venv_clean\Lib\site-packages\keras\src\backend\tensorflow\nn.py:1216: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 204ms/step - accuracy: 0.1271 - loss: 1.1355 - val_accuracy: 0.1765 - val_loss: 1.1384
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.1304 - loss: 1.1252 - val_accuracy: 0.0882 - val_loss: 1.1319
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.2441 - loss: 1.1186 - val_accuracy: 0.3824 - val_loss: 1.1264
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.4448 - loss: 1.1132 - val_accuracy: 0.3824 - val_loss: 1.1214
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.4448 - loss: 1.1081 - val_accuracy: 0.3824 - val_loss: 1.1165
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.4448 - loss: 1.1032 - val_accuracy: 0.3824 - val_loss: 1.1123
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.4448 - loss: 1.0986 - val_accuracy: 0.3824 - val_loss: 1.1079
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.4448 - loss: 1.0948 - val_accuracy: 0.3824 - val_loss: 1.1043
Epo

In [14]:
model_logit_true = keras.Model(inputs=inputs, outputs=outputs, name="penguin_model_scaled")

model_logit_true.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

history_logit_true = model_logit_true.fit(scaled_penguins_x, penguins_y, batch_size = 64, epochs = 100, validation_split = 0.1)

scores = model_logit_true.evaluate(scaled_penguins_x, penguins_y, verbose = 2)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.7960 - loss: 0.4184 - val_accuracy: 0.7941 - val_loss: 0.3869
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7960 - loss: 0.4084 - val_accuracy: 0.7941 - val_loss: 0.3780
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7960 - loss: 0.4001 - val_accuracy: 0.7941 - val_loss: 0.3709
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7960 - loss: 0.3939 - val_accuracy: 0.7941 - val_loss: 0.3642
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7960 - loss: 0.3882 - val_accuracy: 0.7941 - val_loss: 0.3577
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7960 - loss: 0.3823 - val_accuracy: 0.7941 - val_loss: 0.3512
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7960 - loss: 0.3764 - val_accuracy: 0.7941 - val_loss: 0.3447
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7960 - loss: 0.3707 - val_accuracy: 0.7941 - val_loss:

In [15]:
model_logit_true.predict(scaled_penguins_x)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 


array([[9.96885538e-01, 9.44543362e-01, 1.24828899e-02],
       [1.21970914e-01, 8.23972464e-01, 5.87416530e-01],
       [9.96758878e-01, 9.57440257e-01, 1.17469709e-02],
       [8.63278692e-04, 2.39080161e-01, 9.70543265e-01],
       [9.96541619e-01, 9.50634778e-01, 1.28137432e-02],
       [5.25843501e-01, 9.43404853e-01, 2.68571556e-01],
       [9.12683725e-01, 9.56843436e-01, 8.67891237e-02],
       [3.38708889e-03, 3.64645094e-01, 9.37117338e-01],
       [6.60610618e-04, 2.29315028e-01, 9.73994136e-01],
       [2.45210063e-03, 3.09064060e-01, 9.49533999e-01],
       [9.11611736e-01, 9.45889473e-01, 9.34874043e-02],
       [9.99970436e-01, 9.75064933e-01, 5.60162938e-04],
       [9.65310335e-01, 9.49100673e-01, 5.21257818e-02],
       [9.99444604e-01, 9.63499367e-01, 3.81322042e-03],
       [6.66734517e-01, 9.51605082e-01, 2.13578537e-01],
       [9.96514499e-01, 9.47754383e-01, 1.31116714e-02],
       [9.99496579e-01, 9.53130901e-01, 3.89119121e-03],
       [9.60380363e-04, 2.43049

In [16]:
penguins['species']

0         Adelie
1      Chinstrap
2         Adelie
3         Gentoo
4         Adelie
         ...    
328       Gentoo
329       Adelie
330       Adelie
331    Chinstrap
332       Adelie
Name: species, Length: 333, dtype: str

In [17]:
penguins_y

array([0, 1, 0, 2, 0, 1, 1, 2, 2, 2, 0, 0, 1, 0, 1, 0, 0, 2, 0, 1, 0, 0,
       1, 2, 0, 0, 2, 1, 2, 1, 2, 1, 0, 0, 1, 1, 2, 2, 0, 0, 0, 0, 2, 2,
       0, 0, 1, 0, 0, 1, 0, 2, 2, 0, 0, 2, 0, 0, 2, 2, 1, 1, 1, 0, 0, 1,
       0, 2, 0, 1, 0, 0, 2, 1, 2, 2, 0, 0, 0, 2, 0, 0, 2, 0, 1, 2, 0, 1,
       2, 2, 2, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 1, 1, 0, 2, 0, 2, 2, 0, 2,
       0, 1, 0, 2, 2, 2, 0, 2, 0, 2, 0, 2, 1, 0, 0, 1, 0, 0, 0, 2, 0, 0,
       2, 0, 0, 0, 2, 0, 1, 0, 0, 2, 0, 1, 2, 1, 2, 1, 2, 2, 2, 2, 0, 0,
       2, 2, 2, 0, 2, 2, 0, 1, 1, 1, 2, 2, 2, 2, 2, 0, 0, 2, 1, 0, 1, 1,
       0, 0, 0, 0, 1, 0, 2, 1, 0, 2, 2, 0, 1, 0, 1, 0, 2, 0, 2, 0, 0, 0,
       2, 0, 2, 0, 1, 0, 0, 2, 2, 2, 0, 0, 0, 2, 2, 0, 0, 1, 0, 2, 0, 1,
       1, 1, 0, 2, 1, 2, 2, 0, 2, 0, 0, 2, 0, 2, 0, 2, 1, 0, 1, 2, 1, 0,
       2, 2, 2, 0, 0, 0, 2, 2, 2, 1, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 1,
       1, 2, 1, 2, 2, 2, 1, 2, 1, 1, 1, 2, 2, 0, 2, 2, 2, 0, 0, 0, 0, 0,
       2, 0, 2, 0, 0, 2, 2, 0, 0, 1, 2, 1, 0, 1, 2,

In [18]:
# now do the same for logit false, name model "model_logit_false"
model_logit_false = keras.Model(inputs=inputs, outputs=outputs, name="model_logit_false")
model_logit_false.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)
history_logit_false = model_logit_false.fit(scaled_penguins_x, penguins_y, batch_size = 64, epochs = 100, validation_split = 0.1)
scores = model_logit_false.evaluate(scaled_penguins_x, penguins_y, verbose = 2)

Epoch 1/100


c:\Users\ivpri\gsb545\.venv_clean\Lib\site-packages\keras\src\backend\tensorflow\nn.py:1216: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.9565 - loss: 0.1141 - val_accuracy: 1.0000 - val_loss: 0.0444
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9565 - loss: 0.1114 - val_accuracy: 1.0000 - val_loss: 0.0435
Epoch 3/100
1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9688 - loss: 0.0990

c:\Users\ivpri\gsb545\.venv_clean\Lib\site-packages\keras\src\backend\tensorflow\nn.py:1216: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9532 - loss: 0.1108 - val_accuracy: 1.0000 - val_loss: 0.0428
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9565 - loss: 0.1100 - val_accuracy: 1.0000 - val_loss: 0.0421
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9565 - loss: 0.1079 - val_accuracy: 1.0000 - val_loss: 0.0418
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9565 - loss: 0.1085 - val_accuracy: 1.0000 - val_loss: 0.0411
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9532 - loss: 0.1062 - val_accuracy: 1.0000 - val_loss: 0.0406
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9565 - loss: 0.1075 - val_accuracy: 1.0000 - val_loss: 0.0401
Epoch 9/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.9565 - loss: 0.1044 - val_accuracy: 1.0000 - val_loss: 0.0396
Epoch 10/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.9599 - loss: 0.1050 - val_accuracy: 1.0000 - val_loss: 0.0397
Epo

In [19]:
model_logit_false.predict(scaled_penguins_x)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 


array([[9.99528110e-01, 9.72282469e-01, 8.70427873e-04],
       [2.60105226e-02, 9.15998757e-01, 5.77111602e-01],
       [9.99564588e-01, 9.69048679e-01, 8.74163990e-04],
       [4.15164577e-05, 2.88946092e-01, 9.92863894e-01],
       [9.99544501e-01, 9.61549163e-01, 1.01236708e-03],
       [3.17385852e-01, 9.74584758e-01, 1.66166380e-01],
       [9.11671162e-01, 9.79715884e-01, 2.65698787e-02],
       [4.18658747e-04, 4.08543766e-01, 9.75281000e-01],
       [6.87785432e-05, 3.01271290e-01, 9.90938485e-01],
       [1.97833040e-04, 3.91173422e-01, 9.82421339e-01],
       [9.73487854e-01, 9.63295400e-01, 1.55846858e-02],
       [9.99999285e-01, 9.75241363e-01, 9.28996906e-06],
       [9.83816028e-01, 9.79950249e-01, 8.08524340e-03],
       [9.99974072e-01, 9.80132639e-01, 1.02115933e-04],
       [5.52902460e-01, 9.85056221e-01, 8.79083574e-02],
       [9.99605834e-01, 9.72188771e-01, 7.71838007e-04],
       [9.99973714e-01, 9.72381532e-01, 1.22646627e-04],
       [4.84421907e-05, 2.92681

In [20]:
penguins_y

array([0, 1, 0, 2, 0, 1, 1, 2, 2, 2, 0, 0, 1, 0, 1, 0, 0, 2, 0, 1, 0, 0,
       1, 2, 0, 0, 2, 1, 2, 1, 2, 1, 0, 0, 1, 1, 2, 2, 0, 0, 0, 0, 2, 2,
       0, 0, 1, 0, 0, 1, 0, 2, 2, 0, 0, 2, 0, 0, 2, 2, 1, 1, 1, 0, 0, 1,
       0, 2, 0, 1, 0, 0, 2, 1, 2, 2, 0, 0, 0, 2, 0, 0, 2, 0, 1, 2, 0, 1,
       2, 2, 2, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 1, 1, 0, 2, 0, 2, 2, 0, 2,
       0, 1, 0, 2, 2, 2, 0, 2, 0, 2, 0, 2, 1, 0, 0, 1, 0, 0, 0, 2, 0, 0,
       2, 0, 0, 0, 2, 0, 1, 0, 0, 2, 0, 1, 2, 1, 2, 1, 2, 2, 2, 2, 0, 0,
       2, 2, 2, 0, 2, 2, 0, 1, 1, 1, 2, 2, 2, 2, 2, 0, 0, 2, 1, 0, 1, 1,
       0, 0, 0, 0, 1, 0, 2, 1, 0, 2, 2, 0, 1, 0, 1, 0, 2, 0, 2, 0, 0, 0,
       2, 0, 2, 0, 1, 0, 0, 2, 2, 2, 0, 0, 0, 2, 2, 0, 0, 1, 0, 2, 0, 1,
       1, 1, 0, 2, 1, 2, 2, 0, 2, 0, 0, 2, 0, 2, 0, 2, 1, 0, 1, 2, 1, 0,
       2, 2, 2, 0, 0, 0, 2, 2, 2, 1, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 1,
       1, 2, 1, 2, 2, 2, 1, 2, 1, 1, 1, 2, 2, 0, 2, 2, 2, 0, 0, 0, 0, 0,
       2, 0, 2, 0, 0, 2, 2, 0, 0, 1, 2, 1, 0, 1, 2,